# Common Data Preprocessing — UCI HAR Dataset

This notebook creates the common processed dataset used by all four
deep-learning architectures in the project: MLP, 1D CNN, LSTM and GRU.

The preprocessing pipeline is designed to ensure experimental fairness
and prevent data leakage.

The main steps are:

- loading the raw inertial sensor signals,
- encoding activity labels,
- loading participant identifiers,
- creating a subject-wise validation partition,
- verifying participant separation,
- standardizing sensor channels using training statistics only,
- converting arrays to model-compatible data types, and
- saving the final processed dataset.

Architecture-specific transformations such as MLP flattening are not
performed here.

## 1. Imports and Reproducibility

NumPy is used for numerical processing and `GroupShuffleSplit` is used
to create a validation partition while preserving participant groups.

A fixed random seed of 42 is used so that the subject-wise split can be
reproduced consistently.

In [1]:
# Numerical processing
import numpy as np
# Create a validation split while keeping subjects grouped together
from sklearn.model_selection import GroupShuffleSplit
# Fixed seed for reproducible splitting
SEED = 42

## 2. Dataset and Output Paths

The notebook dynamically locates the raw UCI HAR dataset depending on
the directory from which the notebook is executed.

A shared `processed_Data` directory is also created at the project root.
The final preprocessed dataset will be stored in this directory so that
all four model implementations can load exactly the same data.

In [2]:
from pathlib import Path

CURRENT_DIR = Path.cwd()

# Possible dataset locations depending on
# where VS Code starts the notebook
candidates = [
    CURRENT_DIR / "Data" / "UCI HAR Dataset",
    CURRENT_DIR.parent / "Data" / "UCI HAR Dataset"
]

DATA_DIR = None

for path in candidates:
    if path.exists():
        DATA_DIR = path
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate the UCI HAR Dataset folder."
    )

# DATA_DIR is:
# project/Data/UCI HAR Dataset
#
# Therefore project root is two levels above DATA_DIR
PROJECT_ROOT = DATA_DIR.parent.parent

# Folder where processed data will be saved
OUTPUT_DIR = PROJECT_ROOT / "processed_Data"

# Create it if it does not exist
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)
print("Output directory exists:", OUTPUT_DIR.exists())

Current directory: d:\4Y\Y4 S2\DL\Assignment\Git Repo\Deep-Learning-Assignment\Preprocessing
Project root: d:\4Y\Y4 S2\DL\Assignment\Git Repo\Deep-Learning-Assignment
Dataset directory: d:\4Y\Y4 S2\DL\Assignment\Git Repo\Deep-Learning-Assignment\Data\UCI HAR Dataset
Dataset exists: True
Output directory: d:\4Y\Y4 S2\DL\Assignment\Git Repo\Deep-Learning-Assignment\processed_Data
Output directory exists: True


## 3. Loading the Raw Inertial Signals

The nine raw sensor channels are loaded for the official UCI training
and testing partitions.

The channels are combined into arrays with shape:

`(samples, 128, 9)`

The sequence structure is preserved because CNN, LSTM and GRU models
require the temporal dimension. Flattening for the MLP is performed
later inside the MLP implementation rather than as part of the common
preprocessing.

In [3]:
SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


def load_signals(base_path, split):
    signal_data = []

    for signal in SIGNALS:
        file_path = (
            base_path
            / split
            / "Inertial Signals"
            / f"{signal}_{split}.txt"
        )

        signal_data.append(
            np.loadtxt(file_path)
        )

    return np.transpose(
        np.array(signal_data),
        (1, 2, 0)
    )

In [4]:
ACTIVITY_NAMES = np.array([
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
])

In [5]:
X_train_full = load_signals(
    DATA_DIR,
    "train"
)

X_test = load_signals(
    DATA_DIR,
    "test"
)

print(X_train_full.shape)
print(X_test.shape)

(7352, 128, 9)
(2947, 128, 9)


In [6]:
y_train_full = np.loadtxt(
    DATA_DIR / "train" / "y_train.txt",
    dtype=int
)

y_test = np.loadtxt(
    DATA_DIR / "test" / "y_test.txt",
    dtype=int
)

## 4. Activity Label Encoding

The original UCI HAR activity labels use integer values from 1 to 6.

For neural-network training, the labels are converted to zero-based
class indices from 0 to 5 by subtracting one from every label.

This representation is suitable for the sparse categorical
cross-entropy loss function used by the classification models.

In [7]:
# Convert UCI labels 1-6 into zero-based class labels 0-5
y_train_full = y_train_full - 1
y_test = y_test - 1

print(np.unique(y_train_full))
print(np.unique(y_test))

[0 1 2 3 4 5]
[0 1 2 3 4 5]


## 5. Loading Participant Identifiers

Participant identifiers are loaded for both official partitions.

These identifiers are not used as model features. They are used only to
create and verify participant-independent data partitions.

In [8]:
subjects_train_full = np.loadtxt(
    DATA_DIR / "train" / "subject_train.txt",
    dtype=int
)

subjects_test = np.loadtxt(
    DATA_DIR / "test" / "subject_test.txt",
    dtype=int
)

## 6. Creating the Validation Partition

The official UCI test partition is preserved unchanged for final model
evaluation.

A validation subset is created only from the original training
partition using `GroupShuffleSplit`. Subject identifiers are supplied
as grouping variables, which ensures that all windows belonging to a
participant are assigned entirely to either the training or validation
partition.

A validation proportion of 20% and a fixed random seed of 42 are used.

In [9]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        X_train_full,
        y_train_full,
        groups=subjects_train_full
    )
)

In [10]:
X_train = X_train_full[train_idx]
X_val = X_train_full[val_idx]

y_train = y_train_full[train_idx]
y_val = y_train_full[val_idx]

subjects_train = subjects_train_full[train_idx]
subjects_val = subjects_train_full[val_idx]

In [11]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (5551, 128, 9)
Validation: (1801, 128, 9)
Testing: (2947, 128, 9)


### Resulting Data Partitions

The resulting partitions contain:

- Training: 5,551 windows
- Validation: 1,801 windows
- Test: 2,947 windows

Each sample retains the `(128, 9)` sensor representation.

## 7. Verification of Subject Independence

The unique participant IDs in the training, validation and test
partitions are compared after splitting.

The expected condition is:

`Train ∩ Validation = ∅`

`Train ∩ Test = ∅`

`Validation ∩ Test = ∅`

This verification is important because participant overlap could produce
optimistically biased validation or test performance.

In [12]:
train_subjects = set(
    np.unique(subjects_train)
)

val_subjects = set(
    np.unique(subjects_val)
)

test_subjects = set(
    np.unique(subjects_test)
)

print(
    "Train ∩ Validation:",
    train_subjects & val_subjects
)

print(
    "Train ∩ Test:",
    train_subjects & test_subjects
)

print(
    "Validation ∩ Test:",
    val_subjects & test_subjects
)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()


In [13]:
assert train_subjects.isdisjoint(
    val_subjects
), "Train and validation subjects overlap"

assert train_subjects.isdisjoint(
    test_subjects
), "Train and test subjects overlap"

assert val_subjects.isdisjoint(
    test_subjects
), "Validation and test subjects overlap"

print(
    "Subject split verified: "
    "no overlap between train, validation and test."
)

Subject split verified: no overlap between train, validation and test.


### Observation

No participant overlap was detected between the training, validation
and test partitions. The resulting experimental setup therefore
evaluates the ability of the models to generalize to unseen
participants.

## 8. Activity Distribution After Splitting

The activity distribution is checked again after creating the
subject-wise validation partition.

Because participants rather than individual windows are separated,
class proportions are not guaranteed to remain exactly identical.
Therefore, this check verifies that all six activity classes remain
represented in the training, validation and test datasets.

In [14]:
def print_split_distribution(
    name,
    labels
):
    counts = np.bincount(
        labels,
        minlength=6
    )

    percentages = (
        counts /
        len(labels) *
        100
    )

    print(f"\n{name}")

    for i, activity in enumerate(
        ACTIVITY_NAMES
    ):
        print(
            f"{activity:20s} "
            f"{counts[i]:5d} "
            f"({percentages[i]:5.2f}%)"
        )


print_split_distribution(
    "Training",
    y_train
)

print_split_distribution(
    "Validation",
    y_val
)

print_split_distribution(
    "Testing",
    y_test
)


Training
Walking                888 (16.00%)
Walking Upstairs       797 (14.36%)
Walking Downstairs     744 (13.40%)
Sitting                993 (17.89%)
Standing              1053 (18.97%)
Laying                1076 (19.38%)

Validation
Walking                338 (18.77%)
Walking Upstairs       276 (15.32%)
Walking Downstairs     242 (13.44%)
Sitting                293 (16.27%)
Standing               321 (17.82%)
Laying                 331 (18.38%)

Testing
Walking                496 (16.83%)
Walking Upstairs       471 (15.98%)
Walking Downstairs     420 (14.25%)
Sitting                491 (16.66%)
Standing               532 (18.05%)
Laying                 537 (18.22%)


### Observation

All six activities remain represented in all three partitions. Although
the class percentages vary slightly, no class is severely
underrepresented or absent.

## 9. Channel-Wise Standardization

Neural networks generally train more reliably when input variables have
comparable numerical scales.

For each of the nine sensor channels, a mean and standard deviation are
calculated across all samples and time steps in the training partition.

The standardized value is calculated as:

\[
x_{standardized} =
\frac{x-\mu_{train}}{\sigma_{train}}
\]

The validation and test partitions are transformed using the same
training-derived statistics.

Normalization statistics are never calculated separately from the
validation or test data, preventing information from unseen datasets
from influencing model development.

In [15]:
# Calculate one mean value for each sensor channel using TRAINING data only
mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)
# Calculate training-only standard deviation for each channel
std = X_train.std(
    axis=(0, 1),
    keepdims=True
)
# Protect against division by zero
std = np.where(
    std == 0,
    1,
    std
)

In [16]:
# Apply the SAME training-derived statistics to every partition
X_train_normalized = (
    X_train - mean
) / std

X_val_normalized = (
    X_val - mean
) / std

X_test_normalized = (
    X_test - mean
) / std

## 10. Verification of Normalization

The standardized training data is inspected to verify that each sensor
channel has a mean close to zero and a standard deviation close to one.

The normalized train, validation and test arrays are also checked for
NaN and infinite values to ensure that the transformation did not
introduce invalid numerical values.

In [17]:
for name, X in [
    ("Train", X_train_normalized),
    ("Validation", X_val_normalized),
    ("Test", X_test_normalized)
]:
    assert not np.isnan(X).any(), (
        f"{name} contains NaN values"
    )

    assert not np.isinf(X).any(), (
        f"{name} contains infinite values"
    )

print(
    "Normalization verified: "
    "no NaN or infinite values."
)

Normalization verified: no NaN or infinite values.


## 11. TensorFlow-Compatible Data Types

The normalized input arrays are converted from NumPy's default floating
point representation to `float32`, while the class labels are converted
to `int32`.

Using `float32` reduces memory consumption and is the standard
floating-point representation used for TensorFlow model training.

In [18]:
# Convert sensor data to TensorFlow-friendly 32-bit floating point
X_train_normalized = (
    X_train_normalized.astype(
        np.float32
    )
)

X_val_normalized = (
    X_val_normalized.astype(
        np.float32
    )
)

X_test_normalized = (
    X_test_normalized.astype(
        np.float32
    )
)
# Convert class labels to 32-bit integers
y_train = y_train.astype(
    np.int32
)

y_val = y_val.astype(
    np.int32
)

y_test = y_test.astype(
    np.int32
)

In [19]:
print(
    "X train dtype:",
    X_train_normalized.dtype
)

print(
    "X validation dtype:",
    X_val_normalized.dtype
)

print(
    "X test dtype:",
    X_test_normalized.dtype
)

print(
    "y train dtype:",
    y_train.dtype
)

X train dtype: float32
X validation dtype: float32
X test dtype: float32
y train dtype: int32


In [20]:
print(
    "Training mean:",
    X_train_normalized.mean(
        axis=(0, 1)
    )
)

print(
    "Training std:",
    X_train_normalized.std(
        axis=(0, 1)
    )
)

Training mean: [ 6.7512917e-10 -6.8251133e-10 -1.3422051e-10 -2.7427962e-09
  1.8898247e-09  2.0938400e-10  8.6599075e-09  3.6003311e-08
  1.1537595e-08]
Training std: [1.         0.9999995  1.0000001  0.99999815 1.         0.99999887
 0.9999995  0.99999934 1.0000004 ]


## 12. Saving the Common Processed Dataset

The finalized training, validation and test arrays are stored in a
compressed NumPy archive named `har_processed.npz`.

The same file is used by the MLP, CNN, LSTM and GRU notebooks, ensuring
that every architecture receives identical data partitions and common
preprocessing.

The saved file also contains participant IDs, activity names, sensor
channel names and normalization statistics for reproducibility.

In [21]:
# Save all common model inputs, labels and preprocessing metadata
ACTIVITY_NAMES = np.array([
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
])

output_file = (
    OUTPUT_DIR
    / "har_processed.npz"
)
SIGNAL_NAMES = np.array(
    SIGNALS
)
# Compression reduces storage size without changing numerical values
np.savez_compressed(
    output_file,

    X_train=X_train_normalized,
    X_val=X_val_normalized,
    X_test=X_test_normalized,

    y_train=y_train,
    y_val=y_val,
    y_test=y_test,

    subjects_train=subjects_train,
    subjects_val=subjects_val,
    subjects_test=subjects_test,

    mean=mean,
    std=std,

    activity_names=ACTIVITY_NAMES,
    signal_names=SIGNAL_NAMES
)

print(
    "Saved processed dataset to:",
    output_file
)


Saved processed dataset to: d:\4Y\Y4 S2\DL\Assignment\Git Repo\Deep-Learning-Assignment\processed_Data\har_processed.npz


In [22]:
print("File exists:", output_file.exists())
print(
    "File size:",
    round(output_file.stat().st_size / (1024 * 1024), 2),
    "MB"
)

File exists: True
File size: 23.12 MB


## 13. Verification of the Saved Dataset

The processed file is reloaded immediately after saving to verify that
the expected arrays and metadata are present.

Shapes and class labels are printed as a final integrity check before
the dataset is used by the model notebooks.

In [23]:
check_data = np.load(output_file)

print(check_data.files)

print("X_train:", check_data["X_train"].shape)
print("X_val:", check_data["X_val"].shape)
print("X_test:", check_data["X_test"].shape)

print(
    "Train labels:",
    np.unique(check_data["y_train"])
)

print(
    "Validation labels:",
    np.unique(check_data["y_val"])
)

print(
    "Test labels:",
    np.unique(check_data["y_test"])
)

['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'subjects_train', 'subjects_val', 'subjects_test', 'mean', 'std', 'activity_names', 'signal_names']
X_train: (5551, 128, 9)
X_val: (1801, 128, 9)
X_test: (2947, 128, 9)
Train labels: [0 1 2 3 4 5]
Validation labels: [0 1 2 3 4 5]
Test labels: [0 1 2 3 4 5]
